# SCHISM boundaries and namelists

**Learning goals:** Configure real tidal/ocean boundary inputs and the SCHISM namelist structure.

**Prerequisites:** Lesson 4; shared SCHISM fixtures. Actual model execution remains optional.

**Execution contract:** This lesson is **configuration-only**. Documentation rendering never executes SCHISM, downloads data, or requires MPI/Docker.

## Checkpoint

By the end of this lesson, record what was configured and which steps still require a model runtime.

Previous: [journey_04_schism_forcing](../journey_04_schism_forcing/)

Next: [journey_06_schism_real_case](../journey_06_schism_real_case/)


## Why this matters: SCHISM data preparation

**Without Rompy:** preparing HYCOM boundary conditions can mean downloading a large global dataset, selecting the run period and region, interpolating onto open-boundary nodes, extracting the required variables, and writing `elev2D.th.nc` or other SCHISM files. ERA5 and tidal inputs require similarly separate preparation steps.

**With Rompy:** source objects, grid metadata, time ranges, filters, and boundary mappings are assembled into `SCHISMConfig`. Workspace generation carries out the configured cropping, interpolation, boundary extraction, and SCHISM-format conversion. The modeller still chooses appropriate datasets, variables, coordinates, numerical settings, and scientific validation checks.

The following cells show the source fields, model domain, and generated artefacts so this automation remains inspectable.


In [ ]:
import sys
from pathlib import Path

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "scripts" / "schism_case_data.py").is_file())
sys.path.insert(0, str(root))
from scripts.schism_case_data import ensure_schism_data

case = ensure_schism_data()
print("Fixture directory:", case)


In [ ]:
from rompy.core.source import SourceFile
from rompy_schism.data import SCHISMDataBoundary

# Boundary objects retain the source and variable/coordinate mapping.
elevation = SCHISMDataBoundary(
    id="elev2D",
    source=SourceFile(uri=case / "hycom.nc"),
    variables=["surf_el"],
    coords={"t": "time", "y": "ylat", "x": "xlon"},
)
print("Configured boundary source:", elevation.id)
print("Tidal assets:", case / "tides")


## What boundary preparation replaces

The boundary configuration below describes the source variables and coordinate mapping. Rompy uses the SCHISM mesh to sample those fields at open-boundary nodes and writes SCHISM's boundary formats; the plot makes that source-to-boundary relationship inspectable. This is the verification step for boundary extraction.


In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
from rompy.core.data import DataBlob
from rompy_schism import SCHISMGrid

grid = SCHISMGrid(hgrid=DataBlob(source=case / "hgrid.gr3"), vgrid=DataBlob(source=case / "vgrid.in"), drag=1)
dataset = xr.open_dataset(case / "hycom.nc")
fig, ax = plt.subplots(figsize=(8, 5))
dataset.surf_el.isel(time=0).plot(ax=ax, cmap="BrBG")
x, y = grid.boundary_points()
ax.scatter(x, y, s=10, c="black", label="SCHISM open-boundary nodes")
ax.set_title("HYCOM source field and SCHISM boundary samples")
ax.legend(); plt.show()
dataset.close()
